# La base CNL1 avant toute analyse

Les autres notebooks analysent. Celui-ci **décrit la base** : ce qu'elle couvre, ce
qui y manque, ce qui y est douteux.

Il existe parce que plusieurs faits structurants n'étaient écrits nulle part et ont
été découverts par accident au fil des analyses — un capteur en panne repéré par une
classification, un plan d'expérience quasi-croisé repéré par un test de saison qui
manquait de puissance. Ces choses doivent être connues **avant**, pas après.

Il produit un livrable réutilisable : `logements_suspects.csv`, la liste des
logements à écarter, pour que chaque notebook n'ait plus à redécouvrir les pannes
à sa façon.

**Quatre questions.** Sur quoi porte l'enquête ? Que couvre-t-elle dans le temps ?
Que manque-t-il ? Que faut-il écarter ?


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 60)

DATA = '../data/cln1/Raw'
CO2_MAX = 6000          # butée du capteur
CO2_EXT = 400           # teneur extérieure de référence
SORTIE = 'logements_suspects.csv'


## 1. Sur quoi porte l'enquête

CNL1 n'est pas une collecte opportuniste : c'est un **sondage à probabilités
inégales** sur le parc de logements français. Le fichier `CNL1_LISTE_LOGEMENT.csv`
en porte le plan — région, département, dates de visite, et surtout des **poids de
sondage** que rien dans nos analyses n'utilise jusqu'ici.

C'est une omission qui a des conséquences : tous les pourcentages produits jusqu'à
présent décrivent **l'échantillon**, pas le parc. La dernière cellule de cette
section chiffre l'écart.


In [ ]:
# ── Le fichier est encodé en UTF-16, comme INFOS_BETA
ref = pd.read_csv(f'{DATA}/Identite_Logement/CNL1_LISTE_LOGEMENT.csv',
                  sep=';', encoding='utf-16')
piece = pd.read_csv(f'{DATA}/Identite_Logement/CNL1_LISTE_PIECE.csv',
                    sep=';', encoding='utf-16')
print(f'{len(ref)} logements dans le plan de sondage · '
      f'{ref["REGION"].nunique()} régions · {ref["DEPARTEMENT"].nunique()} départements')
display(ref.head(4))

# ── Les poids : ce que représente un logement de l'échantillon
print(f'\nPoids de sondage POIDS2006')
print(f'  somme       : {ref["POIDS2006"].sum():,.0f} logements représentés')
print(f'  médiane     : {ref["POIDS2006"].median():,.0f}')
print(f'  rapport max/min : {ref["POIDS2006"].max() / ref["POIDS2006"].min():.1f}')
print('Un logement peut donc en représenter dix-sept fois plus qu\'un autre.')
print('Aucune moyenne non pondérée ne décrit le parc.')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
ax = axes[0]
ax.hist(ref['POIDS2006'] / 1000, bins=40, color='#4C72B0')
ax.set_xlabel('POIDS2006 (milliers de logements représentés)')
ax.set_ylabel("logements de l'échantillon")
ax.set_title('Des poids très inégaux', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# ── Échantillon contre parc : la géographie
geo = ref.groupby('REGION').agg(n=('CODELIEU', 'size'), poids=('POIDS2006', 'sum'))
geo['% échantillon'] = 100 * geo['n'] / geo['n'].sum()
geo['% du parc'] = 100 * geo['poids'] / geo['poids'].sum()
geo['écart'] = geo['% échantillon'] - geo['% du parc']
geo = geo.sort_values('% du parc', ascending=False)
ax = axes[1]
t = geo.head(10)
x = np.arange(len(t))
ax.barh(x - .2, t['% échantillon'], height=.4, color='#4C72B0', label='échantillon')
ax.barh(x + .2, t['% du parc'], height=.4, color='#C44E52', label='parc (pondéré)')
ax.set_yticks(x); ax.set_yticklabels(t.index, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('%')
ax.set_title('Les dix premières régions', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()
display(geo.head(8).round(1))


## 2. Ce que la campagne couvre dans le temps

Une phrase manquait à tous les notebooks : **la campagne s'étale sur plus de deux
ans**, et le nombre de logements suivis simultanément varie d'un facteur dix d'un
mois à l'autre.

La conséquence est plus lourde qu'elle n'en a l'air. Chaque logement n'est mesuré
qu'une semaine, donc **dans une seule saison**. Le plan est quasi-croisé : un effet
de saison ne peut pas être distingué d'un effet de logement, sauf sur la poignée de
logements visités deux fois.


In [ ]:
co2 = pd.read_csv(f'{DATA}/Mesures_Confort_Dynamique/CNL1_SERIE_CO2.txt', sep='\t',
                  encoding='latin1',
                  usecols=['CODELIEU', 'IDN_PIECE', 'DATE_HEURE', 'VALEUR'],
                  parse_dates=['DATE_HEURE'])
co2['VALEUR'] = pd.to_numeric(co2['VALEUR'], errors='coerce')
co2 = co2.dropna(subset=['VALEUR'])
co2['DATE'] = co2['DATE_HEURE'].dt.normalize()
co2['chauffe'] = co2['DATE_HEURE'].dt.month.isin([10, 11, 12, 1, 2, 3, 4])

print(f'Période : {co2["DATE"].min().date()} → {co2["DATE"].max().date()}  '
      f'({(co2["DATE"].max() - co2["DATE"].min()).days} jours)')
jours = co2.groupby('CODELIEU')['DATE'].nunique()
print(f'Journées par logement : min {jours.min()} · médiane {jours.median():.0f} · '
      f'max {jours.max()}')
print(f'  {(jours < 5).sum()} logements ont moins de 5 journées')

# ── Le point structurant : la saison est confondue avec le logement
sais = co2.groupby('CODELIEU')['chauffe'].nunique()
print(f'\nLogements mesurés dans LES DEUX saisons : {(sais == 2).sum()} '
      f'sur {len(sais)}  ({100 * (sais == 2).mean():.1f} %)')
print(f'Part des mesures en période de chauffe : {100 * co2["chauffe"].mean():.0f} %')
print('Un test « chauffe contre hors chauffe » compare donc surtout des logements')
print('différents, pas des saisons. Aucune analyse saisonnière n\'est identifiable.')

fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
ax = axes[0]
ax.hist(jours, bins=range(1, jours.max() + 2), color='#4C72B0', align='left')
ax.set_xlabel('journées mesurées'); ax.set_ylabel('logements')
ax.set_title('Durée de suivi par logement', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
par_mois = co2.groupby(co2['DATE_HEURE'].dt.to_period('M'))['CODELIEU'].nunique()
ax.plot(par_mois.index.to_timestamp(), par_mois.values, marker='o', ms=4,
        color='#4C72B0', lw=2)
ax.set_ylabel('logements actifs'); ax.set_xlabel('mois')
ax.set_title('Charge de la campagne au fil du temps', fontweight='bold')
ax.tick_params(axis='x', rotation=30); ax.grid(True, alpha=0.3)

ax = axes[2]
par_m2 = co2.groupby(co2['DATE_HEURE'].dt.month)['CODELIEU'].nunique()
coul = ['#C44E52' if m in (10, 11, 12, 1, 2, 3, 4) else '#4C72B0'
        for m in par_m2.index]
ax.bar(par_m2.index, par_m2.values, color=coul)
ax.set_xticks(range(1, 13)); ax.set_xlabel('mois de l\'année')
ax.set_ylabel('logements'); ax.set_title('Saison de chauffe en rouge', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 3. Ce qui manque, et où cela se perd

Deux choses différentes sont mesurées ici. Les **valeurs manquantes** au sein de
chaque source, et les **pertes de jointure** — les mesures qui n'ont pas de
correspondance quand on rapproche deux fichiers.

La seconde est la plus traître : elle ne se voit pas dans un `isna()`.


In [ ]:
cascade = [('CO2 lu dans le fichier', len(co2))]

co2['tranche'] = (co2['DATE_HEURE'].dt.hour * 60 + co2['DATE_HEURE'].dt.minute) // 10
dup = co2.duplicated(['CODELIEU', 'DATE', 'tranche']).sum()
co2 = co2.drop_duplicates(['CODELIEU', 'DATE', 'tranche'])
cascade.append((f'après retrait de {dup} doublons de créneau', len(co2)))

for nom, fich in [('température', 'CNL1_SERIE_TPT.txt'), ('humidité', 'CNL1_SERIE_HRE.txt')]:
    s = pd.read_csv(f'{DATA}/Mesures_Confort_Dynamique/{fich}', sep='\t',
                    encoding='latin1', usecols=['CODELIEU', 'DATE_HEURE', 'APPAREIL',
                                                'VALEUR'], parse_dates=['DATE_HEURE'])
    tot = len(s)
    s = s[s['APPAREIL'] == 'QTR']
    s['VALEUR'] = pd.to_numeric(s['VALEUR'], errors='coerce')
    s = s.dropna(subset=['VALEUR']).drop_duplicates(['CODELIEU', 'DATE_HEURE'])
    j = co2.merge(s[['CODELIEU', 'DATE_HEURE']].assign(ok=1),
                  on=['CODELIEU', 'DATE_HEURE'], how='left')
    print(f'{nom:<12} {tot:>9,} lignes lues · {len(s):>9,} sur appareil QTR · '
          f'apparie {100 * j["ok"].notna().mean():5.2f} % des mesures de CO2')

sem = pd.read_csv(f'{DATA}/Questionnaires_Comportement/CNL1_SEMAINIER_DATA.txt',
                  sep='\t', encoding='latin1')
sem['DATE'] = pd.to_datetime(sem['DATE'], format='%d/%m/%Y')
sem['tranche'] = sem['CODE_HEURE'].str[2:].astype(int) - 1001
occ = sem.groupby(['CODELIEU', 'DATE', 'tranche']).agg(
    n_manq=('IDN_PIECE', lambda s: s.isna().sum()), n=('IDN_PIECE', 'size'))
j = co2.merge(occ, on=['CODELIEU', 'DATE', 'tranche'], how='left')
cascade.append(('dont couvertes par le semainier', int(j['n'].notna().sum())))
cascade.append(('dont position de tous les occupants connue',
                int(((j['n'].notna()) & (j['n_manq'] == 0)).sum())))

casc = pd.DataFrame(cascade, columns=['étape', 'mesures'])
casc['% du départ'] = 100 * casc['mesures'] / casc['mesures'].iloc[0]
casc['perdu à cette étape'] = -casc['mesures'].diff().fillna(0).astype(int)
display(casc.round(1))
print('Les deux dernières lignes sont des pertes de JOINTURE : la mesure de CO2')
print('existe, mais rien ne dit qui était là. Elles ne sont visibles dans aucun')
print('`isna()` du fichier de CO2.')

# ── Les manquants du questionnaire, logement par logement
q = pd.read_csv(f'{DATA}/Questionnaires_Comportement/CNL1_Q_LOGEMENT_MENAGE_DATA.csv',
                sep=';', encoding='latin1').set_index('CODELIEU')
VARS_Q = {'FC3': 'type de logement', 'KVNT': 'ventilation', 'STRMEN': 'ménage',
          'HSRF': 'surface', 'NPP': 'nombre de pièces',
          'OFJ4': 'ouverture (chauffe)', 'OFN4': 'ouverture (hors chauffe)'}
manq_q = pd.DataFrame({'manquant (%)': [
    100 * pd.to_numeric(q[v], errors='coerce').isna().mean() for v in VARS_Q]},
    index=list(VARS_Q.values()))
# Deux dénominateurs coexistent et donnent deux chiffres pour la même variable :
# la part des MÉNAGES non renseignés, et la part des MESURES qu'ils portent. Les
# autres notebooks travaillant à la mesure, les deux sont donnés ici pour qu'un
# écart n'ait pas l'air d'une contradiction.
manq_q['des mesures (%)'] = [
    100 * co2['CODELIEU'].map(pd.to_numeric(q[v], errors='coerce')).isna().mean()
    for v in VARS_Q]
manq_q.columns = ['des ménages (%)', 'des mesures (%)']
display(manq_q.round(2).style.background_gradient(cmap='Reds', vmin=0, vmax=10)
        .format('{:.2f}').set_caption('Questionnaire logement · 567 ménages · '
                                      '514 383 mesures'))


## 4. Ce qu'il faut écarter

Quatre défauts guettent un capteur de CO2 : la **butée** haute, la **panne franche**
qui fige le signal, la **dérive** qui interdit tout retour au niveau extérieur, et
le **trou** dans la série.

Les trois premiers ont été rencontrés au hasard des analyses. Ils sont dépistés ici
de façon systématique, et la liste produite est écrite sur disque pour que les
autres notebooks la reprennent au lieu de la reconstituer.


In [ ]:
g = co2.groupby('CODELIEU')['VALEUR']
Q = pd.DataFrame({'mesures': g.size(), 'journées': co2.groupby('CODELIEU')['DATE'].nunique(),
                  'p05': g.quantile(.05), 'médiane': g.median(), 'p95': g.quantile(.95)})
Q['amplitude'] = Q['p95'] - Q['p05']
Q['sature (%)'] = 100 * co2.assign(s=co2['VALEUR'] >= CO2_MAX).groupby('CODELIEU')['s'].mean()

# Un pas de mesure supérieur à 20 minutes est une interruption d'enregistrement.
co2 = co2.sort_values(['CODELIEU', 'DATE_HEURE'])
ecart = co2.groupby('CODELIEU')['DATE_HEURE'].diff().dt.total_seconds() / 60
Q['trous'] = co2.assign(t=ecart > 20).groupby('CODELIEU')['t'].sum().astype(int)

# ── Les quatre critères
Q['capteur figé'] = Q['amplitude'] < 100          # ne réagit plus à l'occupation
Q['jamais renouvelé'] = Q['p05'] > 1000           # ne redescend jamais vers l'extérieur
Q['butée fréquente'] = Q['sature (%)'] > 1        # censure une part notable du signal
Q['suivi trop court'] = Q['journées'] < 3         # profil moyen non estimable
CRIT = ['capteur figé', 'jamais renouvelé', 'butée fréquente', 'suivi trop court']

bilan = pd.DataFrame({'logements concernés': [int(Q[c].sum()) for c in CRIT]}, index=CRIT)
bilan['% des 535'] = (100 * bilan['logements concernés'] / len(Q)).round(1)
display(bilan)

Q['motifs'] = Q[CRIT].apply(lambda r: ' + '.join([c for c in CRIT if r[c]]), axis=1)
SUSPECTS = Q[Q[CRIT].any(axis=1)].copy()
print(f'\n{len(SUSPECTS)} logements suspects sur {len(Q)} '
      f'({100 * len(SUSPECTS) / len(Q):.1f} %), '
      f'{SUSPECTS["mesures"].sum():,} mesures concernées')
display(SUSPECTS[['mesures', 'journées', 'p05', 'médiane', 'p95', 'amplitude',
                  'sature (%)', 'motifs']].round(1).sort_values('amplitude'))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
ax = axes[0]
ax.scatter(Q['médiane'], Q['amplitude'], s=18, color='#B8C4CA', edgecolors='none',
           label='logements')
s = Q[Q[CRIT].any(axis=1)]
ax.scatter(s['médiane'], s['amplitude'], s=52, color='#C44E52', edgecolors='black',
           linewidths=.7, zorder=3, label='suspects')
ax.axhline(100, color='#C44E52', ls='--', lw=1.2)
ax.annotate('amplitude < 100 ppm', (Q['médiane'].max(), 115), fontsize=8,
            color='#C44E52', ha='right', fontweight='bold')
ax.set_xlabel('CO2 médian (ppm)'); ax.set_ylabel('amplitude p05–p95 (ppm)')
ax.set_title('Dépistage des capteurs figés', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(Q['p05'], bins=50, color='#4C72B0')
ax.axvline(1000, color='#C44E52', ls='--', lw=1.4)
ax.axvline(CO2_EXT, color='#3F7F53', ls='--', lw=1.4)
ax.annotate('extérieur\n400 ppm', (CO2_EXT, ax.get_ylim()[1] * .8), fontsize=8,
            color='#3F7F53', ha='center', fontweight='bold')
ax.set_xlabel('5ᵉ centile du logement (ppm)'); ax.set_ylabel('logements')
ax.set_title('Le logement redescend-il vers l\'extérieur ?', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Les séries des suspects, pour juger sur pièce plutôt que sur un seuil
ax = axes[2]
for c in SUSPECTS.index[:6]:
    x = co2[co2['CODELIEU'] == c].sort_values('DATE_HEURE')
    ax.plot(np.arange(len(x)) / 144, x['VALEUR'].values, lw=1.1, label=c)
ax.set_xlabel('jours depuis la pose'); ax.set_ylabel('CO2 (ppm)')
ax.set_title('Les six premiers suspects, série brute', fontweight='bold')
ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Butée à {CO2_MAX} ppm : '
      f'{int((co2["VALEUR"] >= CO2_MAX).sum()):,} mesures sur '
      f'{co2.loc[co2["VALEUR"] >= CO2_MAX, "CODELIEU"].nunique()} logements '
      f'({100 * (co2["VALEUR"] >= CO2_MAX).mean():.3f} % du fichier)')
print(f'Interruptions de plus de 20 min : {int(Q["trous"].sum())} sur '
      f'{len(co2):,} intervalles ; pas médian {ecart.median():.0f} min, '
      f'pas maximal {ecart.max():.0f} min.')
print('Les séries sont donc continues : la dérivée par filtre glissant, calculée')
print('dans les autres notebooks, porte sur un signal sans interruption.')


## 5. Le livrable

`logements_suspects.csv` porte un `CODELIEU` par ligne et le motif du signalement.

**Ce n'est pas une liste d'exclusion automatique.** Un logement à 2 300 ppm en
permanence peut être un vrai cas de confinement sévère plutôt qu'un capteur dérivé ;
seule l'inspection de la série tranche, et la figure ci-dessus est là pour ça. La
liste sert à ce que la question soit **posée** dans chaque notebook, pas à ce que la
réponse soit tranchée d'avance.


In [ ]:
suspects = SUSPECTS[['motifs', 'mesures', 'journées', 'p05', 'médiane',
                     'amplitude', 'sature (%)']].round(1)
suspects.to_csv(SORTIE, encoding='utf-8')
print(f'écrit : {SORTIE}  ({len(suspects)} logements)')
print('\nÀ reprendre dans les autres notebooks :')
print("    suspects = pd.read_csv('logements_suspects.csv', index_col='CODELIEU')")
print("    d = d[~d['CODELIEU'].isin(suspects.index)]")
display(suspects)

print('\n── Ce que leur retrait changerait')
# Une nuit dont l'enregistrement commence à 3 h n'a que la moitié de ses
# créneaux : sa moyenne n'est pas comparable aux autres. Combien y en a-t-il,
# et le chiffre d'exposition en dépend-il ?
nuit_s = co2[co2['DATE_HEURE'].dt.hour < 6]
taille_n = nuit_s.groupby(['CODELIEU', 'DATE']).size()
mn = nuit_s.groupby(['CODELIEU', 'DATE'])['VALEUR'].mean()
ml = mn.groupby(level=0).median()
ml_c = mn[taille_n >= 36].groupby(level=0).median()
print(f'Nuits incomplètes (moins de 36 créneaux) : {int((taille_n < 36).sum())} '
      f'sur {len(taille_n):,} ({100 * (taille_n < 36).mean():.1f} %)')
print('  seuil       toutes nuits   nuits complètes seulement')
for s_ in (1000, 1500, 2000):
    print(f'  {s_} ppm         {100 * (ml > s_).mean():5.1f} %'
          f'          {100 * (ml_c > s_).mean():5.1f} %')
print('Écart de 0.3 point au plus : les nuits partielles ne biaisent pas le')
print('résultat, il est inutile de les écarter.')
for s in (1000, 1500, 2000):
    a, b = 100 * (ml > s).mean(), 100 * (ml.drop(index=SUSPECTS.index, errors='ignore') > s).mean()
    print(f'  logements dont la nuit médiane dépasse {s} ppm : '
          f'{a:5.1f} %  →  {b:5.1f} %   ({b - a:+.1f})')

# ── Et ce que la pondération de sondage changerait, elle
w = ref.set_index('CODELIEU')['POIDS2006'].reindex(ml.index)
ok = w.notna()
print('\n── Ce que la pondération de sondage changerait')
for s in (1000, 1500, 2000):
    brut = 100 * (ml > s).mean()
    pond = 100 * np.average((ml[ok] > s).astype(float), weights=w[ok])
    print(f'  {s} ppm : échantillon {brut:5.1f} %  →  parc pondéré {pond:5.1f} %   '
          f'({pond - brut:+.1f})')
print('\nL\'écart est modeste mais systématique, et toujours à la baisse : les')
print('logements les plus chargés sont un peu sur-représentés dans l\'échantillon.')
print('Tout chiffre présenté comme national doit être pondéré.')
